# Выбор следующего токена

Дополнение к § 3.4. Учебные распределения показывают разницу между жадным выбором и сэмплированием. Затем полный опыт сравнивает продолжения русскоязычной модели при разных prompt, лимитах, температурах и seed.


## Запуск полного опыта

Установите пакеты из `requirements.txt` со страницы главы. При первом запуске веса модели загрузятся из интернета; генерация выполняется на CPU. В ячейке «Опыт с двумя началами» установите `run_model = True` и запустите ячейки сверху вниз. Предварительно сформулируйте гипотезу о продолжениях, а после опыта анализируйте таблицу, где сохранены prompt, длина, режим, температура, seed и текст. Учебные числовые упражнения не требуют загрузки весов.


## Один шаг

Проверьте, что доли складываются в 1, и выберите лидера жадным способом. При сэмплировании каждый кандидат может быть выбран с вероятностью, заданной его долей.


In [ ]:
import numpy as np
words = np.array(["фигурку","задачу","вчера"])
prob = np.array([0.6,0.3,0.1])
assert np.isclose(prob.sum(), 1)
print("Жадно:", words[np.argmax(prob)])

## Повторяем случайный выбор

Измените seed и сравните десять выборов. При такой короткой серии частоты могут заметно отличаться от заданных долей.


In [ ]:
rng = np.random.default_rng(7)
chosen = rng.choice(words, size=10, p=prob)
print(chosen.tolist())

## Меняем распределение

Предположите, как изменится лидер после увеличения второго логита, затем вычислите softmax и сравните новый порядок кандидатов.


In [ ]:
logits = np.array([2.0,1.0,0.0])
logits[1] += 1.5
shift = logits - logits.max()
p2 = np.exp(shift)/np.exp(shift).sum()
print(list(zip(words,p2.round(3))))

## Опыт с двумя началами

Сначала предскажите, как различатся продолжения начала о фигурке и начала о модели самолёта. Запустите полный опыт: для обоих prompt модель выдаёт жадные продолжения на 20 и 40 токенов; для первого prompt выполняется сэмплирование при $T=0{,}8$ и $T=1{,}2$ с тремя seed для каждой температуры. Сравнивайте пары, в которых меняется только один параметр.


In [ ]:
run_model = False
if run_model:
    import pandas as pd
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_id = "ai-forever/rugpt3small_based_on_gpt2"
    revision = "a9307e696cd3c5b7f953ff4cb19d76a4d81821d5"
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
    model = AutoModelForCausalLM.from_pretrained(model_id, revision=revision).eval()
    prompts = ["В школьном кружке напечатали фигурку. Она",
               "В школьном кружке напечатали модель самолёта. Она"]
    encoded = tokenizer(prompts[0], return_tensors="pt")
    ids = encoded["input_ids"][0].tolist()
    print("Токены первого начала:")
    print([(i, tokenizer.decode([i])) for i in ids])
    rows = []
    def continue_text(prompt, limit, seed=None, temperature=1.0):
        encoded = tokenizer(prompt, return_tensors="pt")
        options = {"max_new_tokens": limit, "do_sample": seed is not None,
                   "pad_token_id": tokenizer.eos_token_id}
        if seed is not None:
            torch.manual_seed(seed)
            options.update(top_k=20, temperature=temperature)
        with torch.inference_mode():
            output = model.generate(**encoded, **options)
        new_ids = output[0, encoded["input_ids"].shape[1]:]
        return tokenizer.decode(new_ids, skip_special_tokens=True)
    for prompt in prompts:
        for limit in (20, 40):
            rows.append({"prompt": prompt, "limit": limit, "mode": "greedy",
                         "seed": None, "T": None,
                         "text": continue_text(prompt, limit)})
    for temperature in (0.8, 1.2):
        for seed in (7, 11, 23):
            rows.append({"prompt": prompts[0], "limit": 40, "mode": "sample",
                         "seed": seed, "T": temperature,
                         "text": continue_text(prompts[0], 40, seed, temperature)})
    report = pd.DataFrame(rows)
    print(report.to_string(index=False))


## Разбор результата

В таблице сравните разные начала при одинаковой длине и жадном выборе, затем 20 и 40 токенов для одного начала. Для сэмплирования сопоставьте одинаковые seed при двух температурах и разброс внутри каждой температуры. Отметьте обрывы и повторы. Если есть проверяемое утверждение, найдите независимый источник; если его нет, запишите «проверяемых фактов нет». Для повторения опыта сохраните идентификатор и ревизию модели, версии библиотек, prompt, seed и параметры генерации.


In [ ]:
if run_model:
    again = continue_text(prompts[0], 40, seed=7, temperature=0.8)
    first = report[(report["seed"] == 7) & (report["T"] == 0.8)].iloc[0]["text"]
    print("Повтор с тем же seed:", again == first)
    print("Если в продолжении нет проверяемого факта, отметьте это в отчёте.")


In [ ]:
if run_model:
    from importlib.metadata import version
    print("Библиотеки:", {name: version(name) for name in
          ["torch", "torchvision", "transformers", "numpy", "pandas"]})
    print("Модель и ревизия:", model_id, revision)


## Самопроверка




### 1. Как меняется вход модели после выбора очередного токена?

A. К входной последовательности добавляется выбранный токен.

B. Вход заменяется только последним символом.

C. Модель получает будущее продолжение целиком.

D. Веса модели обучаются на одном ответе.




### 2. Для долей 0,60; 0,30; 0,10 какой токен выберет жадный режим?

A. Первый, с долей 0,60.

B. Второй, с долей 0,30.

C. Третий, с долей 0,10.

D. Выбор всегда случайный.




### 3. Для логитов $(2,1,0)$ при $T=2$ после softmax получены доли примерно $(0{,}506; 0{,}307; 0{,}186)$. Что изменилось относительно $T=1$?

A. Первый кандидат перестал быть лидером.

B. Доли стали ближе друг к другу, лидер сохранился.

C. Все кандидаты получили одинаковую вероятность.

D. Модель добавила новый токен.




### 4. Что ограничивает параметр `max_new_tokens=40`?

A. Число новых токенов, добавленных к prompt.

B. Общую длину текста в словах.

C. Число предложений в ответе.

D. Длину словаря модели.




### 5. В таблице две генерации имеют одинаковые prompt и настройки, но разные seed. Какое различие ожидаемо?

A. При сэмплировании ответы могут различаться из-за случайного выбора.

B. Seed меняет обученные веса модели.

C. Один seed проверяет факты, другой — грамматику.

D. Оба ответа обязаны совпасть.




### 6. Вы сравниваете влияние длины: у одного ответа 20 новых токенов, у другого 40. Что нужно зафиксировать?

A. Начало текста, модель и режим генерации.

B. Только длину prompt.

C. Для каждой длины выбрать новую модель.

D. Одновременно менять начало и температуру.




### 7. В опыте при $T=0{,}8$ и $T=1{,}2$ использованы seed 7, 11 и 23. Как лучше сравнить серии?

A. Сопоставить одинаковые seed между температурами и отдельно посмотреть разброс внутри каждой серии.

B. Сравнить только самый длинный ответ.

C. Считать большее число слов признаком большей точности.

D. Смешать ответы и не записывать температуру.




### 8. В продолжении нет даты, числа или другого проверяемого утверждения. Что записать в отчёте?

A. «Проверяемых фактов нет».

B. «Все утверждения модели верны».

C. Придумать факт, чтобы выполнить проверку.

D. Считать стиль доказательством точности.




### 9. Модель выдала фразу «Кружок открылся в 2018 году». Как проверить её?

A. Найти независимый надёжный источник и записать, подтверждает ли он дату.

B. Считать дату верной, если фраза звучит уверенно.

C. Попросить ту же модель повторить ответ и считать совпадение доказательством.

D. Увеличить температуру.




### 10. Какие сведения нужны, чтобы повторить опыт генерации?

A. Модель и её ревизия, prompt, версии библиотек и параметры генерации, включая seed.

B. Только seed.

C. Только скопированный ответ.

D. Число слов и имя файла.




## Подсказки и решения

### Вопрос 1

**Подсказка**


Следующий шаг учитывает уже созданное начало.



**Решение**


**A.** Модель снова оценивает продолжение для расширенной последовательности токенов.



### Вопрос 2

**Подсказка**


Жадный режим выбирает наибольшую долю.



**Решение**


**A.** Максимальная доля равна 0,60.



### Вопрос 3

**Подсказка**


При $T=2$ логиты перед softmax делят на два.



**Решение**


**B.** Температура сгладила распределение, но наибольшую долю по-прежнему имеет первый кандидат.



### Вопрос 4

**Подсказка**


Параметр считает токены продолжения, а не слова.



**Решение**


**A.** Исходные токены prompt в это ограничение не входят.



### Вопрос 5

**Подсказка**


Seed задаёт начальное состояние генератора случайных чисел.



**Решение**


**A.** При сэмплировании разные seed могут привести к разным токенам и продолжениям.



### Вопрос 6

**Подсказка**


Чтобы связать результат с длиной, остальные условия должны совпадать.



**Решение**


**A.** Тогда обрыв или развитие фразы можно сопоставить с лимитом токенов.



### Вопрос 7

**Подсказка**


Одинаковый seed образует пару условий для сравнения температуры.



**Решение**


**A.** Парное сравнение помогает увидеть влияние температуры, а разброс показывает вариацию между seed.



### Вопрос 8

**Подсказка**


Запишите то, что действительно можно проверить в выходе.



**Решение**


**A.** Отсутствие проверяемого утверждения не подтверждает достоверность модели.



### Вопрос 9

**Подсказка**


Генератор продолжает текст, а не сверяет его с источником.



**Решение**


**A.** Дату нужно проверить по независимому источнику.



### Вопрос 10

**Подсказка**


Повтор зависит и от весов, и от входа, и от настроек среды.



**Решение**


**A.** Полная запись условий позволяет воспроизвести эксперимент насколько это допускает та же среда.



